In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability

In [2]:
Logger().init_logger(None, None, logging_level="WARNING")
animal_ids = [6]
paradigm = [1100]
session_range = [0,33]
session_ids = None
normalize = True
smooth = False
load_fr_track = False
cast_numeric_float32 = True
excl_session_names = ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

In [3]:
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
if cast_numeric_float32:
    num_cols = fr.select_dtypes(include=[np.number]).columns
    fr[num_cols] = fr[num_cols].astype(np.float32)

fr_track = None
if load_fr_track:
    fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
    if cast_numeric_float32:
        num_cols = fr_track.select_dtypes(include=[np.number]).columns
        fr_track[num_cols] = fr_track[num_cols].astype(np.float32)

fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names) # z-scoring within sessions
if cast_numeric_float32:
    num_cols = fr_z_scored.select_dtypes(include=[np.number]).columns
    fr_z_scored[num_cols] = fr_z_scored[num_cols].astype(np.float32)

# fr_z_all_sess is expensive and unused in this notebook; compute only if needed
# ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
# ens_data = analytics.get_analytics('TrackwiseEnsembleProj', session_names=session_names)
ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)#.drop("to_ephys_timestamp", axis=1)
if cast_numeric_float32:
    num_cols = ensamble_proj.select_dtypes(include=[np.number]).columns
    ensamble_proj[num_cols] = ensamble_proj[num_cols].astype(np.float32)
behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))
#ensamble_proj.set_index(['session_id'], inplace = True)
# ens_data.set_index(['session_id'], inplace = True)

/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


In [4]:
counts = fr["Unit0002"].value_counts()
percentages = fr["Unit0002"].value_counts(normalize=True) * 100

result = pd.DataFrame({
    "count": counts,
    "percentage": percentages
})
result

# checking special frs in track and normal df
# count = (fr == 25).to_numpy().sum()
# percentage = (fr == 25).to_numpy().sum() / fr.size * 100
# print(f'occurence count: {count}, percentage: {percentage:.2f}%')
# print(len(fr))

# count = (fr_track == 156.250000).to_numpy().sum()
# percentage = (fr_track == 156.25).to_numpy().sum() / fr_track.size * 100
# print(f'occurence count: {count}, percentage: {percentage:.2f}%')
# print(len(fr_track))

,count,percentage
Unit0002,,
0.0,1155174,87.679973
25.0,150340,11.411101
50.0,11140,0.845548
75.0,752,0.057078
100.0,70,0.005313
125.0,11,0.000835
150.0,2,0.000152


In [5]:
session_counts = (
    fr.groupby(level="session_id")
    .size()
    .rename("n_entries")
    .reset_index()
)
# convert number of bins → minutes
session_counts["duration_min"] = session_counts["n_entries"] * 0.04 / 60
session_counts["session_date"] = session_counts["session_id"].str.split("_").str[0]
session_counts = session_counts.sort_values("session_id")
duplicate_session_dates = session_counts["session_date"].duplicated(keep=False)
session_date_number = session_counts.groupby("session_date").cumcount() + 1
session_counts["session_plot_label"] = np.where(
    duplicate_session_dates,
    session_counts["session_date"] + " (" + session_date_number.astype(str) + ")",
    session_counts["session_date"],
)
long_session_color = "#215CAF"
other_session_color = "#B7352D"
session_counts["duration_color"] = np.where(
    session_counts["duration_min"] > 10,
    long_session_color,
    other_session_color,
)

fig = px.bar(
    session_counts,
    x="session_plot_label",
    y="duration_min",
    labels={
        "session_plot_label": "Session",
        "duration_min": "Recording length (min)",
    },
    title="Recording length per session",
)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=session_counts["session_plot_label"],
    showgrid=False,
    title_standoff=4,
    tickangle=-45,
    tickfont=dict(size=7, family="Arial"),
    title_font=dict(size=8, family="Arial"),
)

y_tick_step = 10
y_tick_max = max(10, int(np.ceil(session_counts["duration_min"].max() / y_tick_step) * y_tick_step))
y_tickvals = np.arange(0, y_tick_max + y_tick_step, y_tick_step)
y_ticktext = [
    "<span style='color:#B7352D'>10</span>" if tick == 10 else f"{tick:g}"
    for tick in y_tickvals
]

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(128,128,128,0.35)",
    gridwidth=0.5,
    tickmode="array",
    tickvals=y_tickvals,
    ticktext=y_ticktext,
    tickfont=dict(size=7, family="Arial"),
    title_font=dict(size=8, family="Arial"),
)

fig.update_traces(marker_color=session_counts["duration_color"].tolist())

fig.add_hline(y=10, line_dash="dash", line_color="#B7352D", line_width=1)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    title=dict(text="Recording length per session", font=dict(size=8, family="Arial")),
    font=dict(family="Arial", size=7),
    width=600,
    height=270,
    margin=dict(l=50, r=20, t=30, b=90),
)

fig.show()


In [6]:
behav.index.get_level_values('session_id').unique()

Index(['2024-11-14_15-01', '2024-11-14_16-40', '2024-11-15_15-48',
       '2024-11-20_17-46', '2024-11-21_17-22', '2024-11-22_16-01',
       '2024-11-25_16-25', '2024-11-26_16-39', '2024-11-27_16-11',
       '2024-11-28_17-41', '2024-12-02_16-09', '2024-12-03_16-23',
       '2024-12-04_18-06', '2024-12-06_16-49', '2024-12-09_17-45',
       '2024-12-10_17-20', '2024-12-11_17-42', '2024-12-12_16-13',
       '2024-12-13_17-10', '2025-01-14_18-08', '2025-01-15_17-18',
       '2025-01-16_17-47', '2025-01-17_16-55', '2025-01-23_16-48',
       '2025-01-24_12-24', '2025-01-24_19-37', '2025-01-25_10-55',
       '2025-01-25_21-29', '2025-01-26_13-45', '2025-01-26_21-48',
       '2025-01-27_13-39'],
      dtype='object', name='session_id')

In [ ]:
res_cue1 = []
res_cue2 = []
d_stop_c1 = []
d_stop_c2 = []
r1_stop = []
r2_stop = []

r1_wrong_only = []
r2_wrong_only = []

for s_id in behav.index.unique("session_id"):
    s_behav = behav.loc[s_id]

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["cue"] == 1)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 1]) * 100
    res_cue1.append((s_id, prop_expert_trials))

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1) &
        (s_behav["cue"] == 2)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 2]) * 100
    res_cue2.append((s_id, prop_expert_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 1]) * 100
    d_stop_c1.append((s_id, prop_d_stop_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 2]) * 100
    d_stop_c2.append((s_id, prop_d_stop_trials))

    r1_wrong = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1)
    ]
    r2_wrong = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["choice_R1"] == 1)
    ]

    r1_wrong_only.append((s_id, len(r1_wrong) / len(s_behav) * 100))
    r2_wrong_only.append((s_id, len(r2_wrong) / len(s_behav) * 100))

    r1_stops = s_behav[s_behav["choice_R1"] == 1]
    r2_stops = s_behav[s_behav["choice_R2"] == 1]

    prop_r1_stop = len(r1_stops) / len(s_behav) * 100
    prop_r2_stop = len(r2_stops) / len(s_behav) * 100
    r1_stop.append((s_id, prop_r1_stop))
    r2_stop.append((s_id, prop_r2_stop))

# numpy conversions
d_stop_c1 = np.array(d_stop_c1, dtype=object)
d_stop_c2 = np.array(d_stop_c2, dtype=object)
res_cue1  = np.array(res_cue1,  dtype=object)
res_cue2  = np.array(res_cue2,  dtype=object)
r1_wrong_only = np.array(r1_wrong_only, dtype=object)
r2_wrong_only = np.array(r2_wrong_only, dtype=object)

idx1 = np.argsort(res_cue1[:, 0])
idx2 = np.argsort(res_cue2[:, 0])

x1, y1 = res_cue1[idx1, 0], res_cue1[idx1, 1].astype(float)
x2, y2 = res_cue2[idx2, 0], res_cue2[idx2, 1].astype(float)
x3, y3 = d_stop_c1[idx1, 0], d_stop_c1[idx1, 1].astype(float)
x4, y4 = d_stop_c2[idx2, 0], d_stop_c2[idx2, 1].astype(float)

x5y5 = np.array(r1_stop, dtype=object)
x5, y5 = x5y5[:, 0], x5y5[:, 1].astype(float)

x6y6 = np.array(r2_stop, dtype=object)
x6, y6 = x6y6[:, 0], x6y6[:, 1].astype(float)

x7, y7 = r1_wrong_only[:, 0], r1_wrong_only[:, 1].astype(float)
x8, y8 = r2_wrong_only[:, 0], r2_wrong_only[:, 1].astype(float)

# date-based split (STRING COMPARISON)
x_pre_cue = "2024-11-27_16-11"

m5 = x5 <= x_pre_cue
x5, y5 = x5[m5], y5[m5]

m6 = x6 <= x_pre_cue
x6, y6 = x6[m6], y6[m6]

m1 = x1 >= x_pre_cue
x1, y1 = x1[m1], y1[m1]

m2 = x2 >= x_pre_cue
x2, y2 = x2[m2], y2[m2]

m3 = x3 >= x_pre_cue
x3, y3 = x3[m3], y3[m3]

m4 = x4 >= x_pre_cue
x4, y4 = x4[m4], y4[m4]

# average expert performance (Cue 1 and Cue 2) per session
cue1_expert = pd.Series(res_cue1[:, 1].astype(float), index=res_cue1[:, 0], name="cue1_expert")
cue2_expert = pd.Series(res_cue2[:, 1].astype(float), index=res_cue2[:, 0], name="cue2_expert")
avg_expert = pd.concat([cue1_expert, cue2_expert], axis=1).mean(axis=1).sort_index()
x_avg_expert = avg_expert.index.to_numpy()
y_avg_expert = avg_expert.to_numpy(dtype=float)
m_avg = x_avg_expert >= x_pre_cue
x_avg_expert, y_avg_expert = x_avg_expert[m_avg], y_avg_expert[m_avg]

# mean Assembly023 activation per session
if "session_id" in ensamble_proj.index.names:
    assembly023_session_mean = ensamble_proj.groupby(level="session_id")["Assembly023"].mean()
elif "session_id" in ensamble_proj.columns:
    assembly023_session_mean = ensamble_proj.groupby("session_id")["Assembly023"].mean()
else:
    raise KeyError("Could not find 'session_id' in ensamble_proj index names or columns")

if "session_id" in behav.index.names:
    behav_sessions = pd.Index(behav.index.get_level_values("session_id").unique()).astype(str)
else:
    behav_sessions = pd.Index(behav.index.unique()).astype(str)

assembly023_session_mean.index = assembly023_session_mean.index.astype(str)
common_sessions = np.intersect1d(behav_sessions.to_numpy(), assembly023_session_mean.index.to_numpy())
assembly023_session_mean = assembly023_session_mean.loc[common_sessions].sort_index()
x_ens023 = assembly023_session_mean.index.to_numpy()
y_ens023 = assembly023_session_mean.to_numpy(dtype=float)

# plotting
fig = go.Figure()

fig.add_trace(go.Scatter(x=x5, y=y5, mode="lines+markers",
    name="R1 Stop Overall",
    line=dict(width=2, color="lightgrey"),
    marker=dict(size=7, color="lightgrey"),
))

fig.add_trace(go.Scatter(x=x6, y=y6, mode="lines+markers",
    name="R2 Stop Overall",
    line=dict(width=2, color="dimgrey"),
    marker=dict(size=7, color="dimgrey"),
))

fig.add_trace(go.Scatter(x=x1, y=y1, mode="lines+markers",
    name="Cue 1 - Expert",
    line=dict(width=2, color="orange"),
    marker=dict(size=7, color="orange"),
))

fig.add_trace(go.Scatter(x=x3, y=y3, mode="lines+markers",
    name="Cue 1 - Double Stop",
    line=dict(width=2, color="peachpuff", dash="dash"),
    marker=dict(size=7, color="peachpuff"),
))

fig.add_trace(go.Scatter(x=x2, y=y2, mode="lines+markers",
    name="Cue 2 - Expert",
    line=dict(width=2, color="purple"),
    marker=dict(size=7, color="purple"),
))

fig.add_trace(go.Scatter(x=x4, y=y4, mode="lines+markers",
    name="Cue 2 - Double Stop",
    line=dict(width=2, color="lavender", dash="dash"),
    marker=dict(size=7, color="lavender"),
))

fig.add_trace(go.Scatter(x=x_avg_expert, y=y_avg_expert, mode="lines+markers",
    name="Average Expert",
    line=dict(width=3, color="red", dash="dot"),
    marker=dict(size=7, color="red"),
))

fig.add_trace(go.Scatter(x=x_ens023, y=y_ens023, mode="lines+markers",
    name="Assembly023 Mean Activation",
    line=dict(width=2, color="blue"),
    marker=dict(size=6, color="blue"),
    yaxis="y2",
))

# fig.add_trace(go.Scatter(
#     x=x7, y=y7,
#     mode="lines+markers",
#     name="C1-Wrong only(R2)",
#     line=dict(width=2, color="lightcoral"),
#     marker=dict(size=7, color="lightcoral"),
# ))

# fig.add_trace(go.Scatter(
#     x=x8, y=y8,
#     mode="lines+markers",
#     name="C2-Wrong only(R1)",
#     line=dict(width=2, color="indianred"),
#     marker=dict(size=7, color="indianred"),
# ))

def to_ordinal(x):
    order = {s: i for i, s in enumerate(sorted(np.unique(x)))}
    return np.array([order[s] for s in x], dtype=float)

x1_num = to_ordinal(x1)
x2_num = to_ordinal(x2)

m2, b2 = np.polyfit(x2_num, y2, 1)
corr1, pval1 = stats.pearsonr(x1_num, y1)
corr2, pval2 = stats.pearsonr(x2_num, y2)

# fig.add_trace(go.Scatter(
#     x=x2, y=m2 * x2_num + b2,
#     mode="lines",
#     name="Cue 2 fit",
#     line=dict(width=2, color="purple", dash="dash"),
# ))

all_x_ticks = np.sort(np.unique(np.concatenate([x5, x6, x1, x2, x3, x4, x_avg_expert, x_ens023])))
all_x_ticklabels = [s.split("_")[0] for s in all_x_ticks]

# layout
fig.update_layout(
    title=dict(text="Animal Performance (Expert Trials and Double Stops)", font=dict(size=10, family="Arial")),
    width=900,
    height=340,
    margin=dict(l=55, r=70, t=35, b=95),
    xaxis_title="Session",
    yaxis_title="Proportion of trials (%)",
    yaxis2=dict(
        title="Assembly023 Mean Activation",
        overlaying="y",
        side="right",
        showgrid=False,
        zeroline=False,
        showline=True,
        title_font=dict(size=10, family="Arial"),
        tickfont=dict(size=9, family="Arial"),
    ),
    plot_bgcolor="white",
    font=dict(family="Arial", size=9),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(0,0,0,0)",
        xanchor="left", yanchor="top",
        orientation="h",
        entrywidth=170,
        entrywidthmode="pixels",
        font=dict(size=9, family="Arial"),
        tracegroupgap=0,
    ),
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.2)",
    zeroline=False,
    showline=True,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)

fig.update_xaxes(
    tickmode="array",
    tickvals=all_x_ticks,
    ticktext=all_x_ticklabels,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.1)",
    showline=True,
    zeroline=False,
    tickangle=-45,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)
fig.update_traces(marker=dict(size=6))

fig.show()


In [8]:
# ensamble_proj already loaded above; avoid re-loading to keep memory stable
ensamble_proj.head()

,session_id,from_ephys_timestamp,to_ephys_timestamp,Assembly001,Assembly002,Assembly003,Assembly004,Assembly005,Assembly006,Assembly007,...,Assembly014,Assembly015,Assembly016,Assembly017,Assembly018,Assembly019,Assembly020,Assembly021,Assembly022,Assembly023
0,2024-11-14_16-40,0.0,40000.0,-0.003749,0.188389,0.596589,-0.271849,0.009416,-0.041533,0.201716,...,0.079650,0.036818,-0.020834,0.177184,-0.069867,-0.103094,0.263283,0.130219,-0.113907,0.772755
1,2024-11-14_16-40,40000.0,80000.0,-0.008523,-0.084321,0.760840,-0.239982,0.010639,0.004561,0.117536,...,0.078771,0.021977,0.231536,0.203376,-0.069608,-0.090912,0.116403,0.151695,0.243103,1.022760
2,2024-11-14_16-40,80000.0,120000.0,-0.008068,0.437550,0.508324,-0.305987,0.002827,-0.072981,0.275745,...,0.075393,0.044933,-0.139091,0.157991,-0.072642,-0.106672,0.381368,0.099889,-0.333252,0.597737
3,2024-11-14_16-40,120000.0,160000.0,0.060204,0.605307,0.464744,-0.253230,0.023734,-0.014145,0.085099,...,0.012177,0.017787,-0.015952,-0.468551,-0.070354,-0.100348,-0.148904,0.156450,0.017423,0.232826
4,2024-11-14_16-40,160000.0,200000.0,0.070604,0.039394,0.683770,-0.258118,0.018634,-0.010309,-0.184203,...,0.040270,0.018259,0.235185,0.187112,-0.062438,-0.036411,-0.130256,0.106865,0.474664,0.024764


In [9]:
# Firing Rate stability grand average -> increasing fr over time? 
fr_units = fr.loc[:, fr.columns.str.startswith("Unit")]
session_mean_hz = fr_units.groupby("session_id").mean()

session_mean_z = (session_mean_hz - session_mean_hz.mean(axis=0)) / session_mean_hz.std(axis=0)

pop_mean_z = session_mean_z.mean(axis=1)

fig = px.line(pop_mean_z, title="Session drift (z-scored session means)")
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()


In [10]:
fr_units = fr_z_scored.loc[:, fr.columns.str.startswith("Unit")]

Z_unit_session = fr_units.groupby("session_id").mean().T  # Units x Sessions

fig = px.imshow(
    Z_unit_session,
    aspect="auto",
    labels=dict(x="Session", y="Unit", color="Mean z"),
    title="Per-neuron drift: session-mean z-score"
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig.show()


In [11]:
if fr_track is None:
    print("fr_track not loaded (load_fr_track=False). Set it to True in the config cell when needed.")
else:
    print(f"fr_track shape: {fr_track.shape}")
    fr_track.head()

fr_track not loaded (load_fr_track=False). Set it to True in the config cell when needed.


In [ ]:
# Animal performance plot copy without Assembly012 overlay; exclude short sessions (<10 trials)
min_trials_per_session = 10

trial_counts = (
    behav.reset_index()[["session_id", "trial_id"]]
    .dropna()
    .drop_duplicates()
    .groupby("session_id")["trial_id"]
    .nunique()
    .sort_index()
)
valid_sessions = pd.Index(trial_counts[trial_counts >= min_trials_per_session].index.astype(str))
excluded_short_sessions = trial_counts[trial_counts < min_trials_per_session].index.astype(str).tolist()

print(f"Excluded sessions with <{min_trials_per_session} trials: {excluded_short_sessions}")

behav_perf = behav.loc[behav.index.get_level_values("session_id").isin(valid_sessions)].copy()

res_cue1 = []
res_cue2 = []
d_stop_c1 = []
d_stop_c2 = []
r1_stop = []
r2_stop = []

r1_wrong_only = []
r2_wrong_only = []

for s_id in behav_perf.index.unique("session_id"):
    s_behav = behav_perf.loc[s_id]

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["cue"] == 1)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 1]) * 100
    res_cue1.append((s_id, prop_expert_trials))

    expert_trials = s_behav[
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1) &
        (s_behav["cue"] == 2)
    ]
    prop_expert_trials = len(expert_trials) / len(s_behav[s_behav["cue"] == 2]) * 100
    res_cue2.append((s_id, prop_expert_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 1]) * 100
    d_stop_c1.append((s_id, prop_d_stop_trials))

    d_stop = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R1"] == 1) &
        (s_behav["choice_R2"] == 1)
    ]
    prop_d_stop_trials = len(d_stop) / len(s_behav[s_behav["cue"] == 2]) * 100
    d_stop_c2.append((s_id, prop_d_stop_trials))

    r1_wrong = s_behav[
        (s_behav["cue"] == 1) &
        (s_behav["choice_R1"] == 0) &
        (s_behav["choice_R2"] == 1)
    ]
    r2_wrong = s_behav[
        (s_behav["cue"] == 2) &
        (s_behav["choice_R2"] == 0) &
        (s_behav["choice_R1"] == 1)
    ]

    r1_wrong_only.append((s_id, len(r1_wrong) / len(s_behav) * 100))
    r2_wrong_only.append((s_id, len(r2_wrong) / len(s_behav) * 100))

    r1_stops = s_behav[s_behav["choice_R1"] == 1]
    r2_stops = s_behav[s_behav["choice_R2"] == 1]

    prop_r1_stop = len(r1_stops) / len(s_behav) * 100
    prop_r2_stop = len(r2_stops) / len(s_behav) * 100
    r1_stop.append((s_id, prop_r1_stop))
    r2_stop.append((s_id, prop_r2_stop))

# numpy conversions
d_stop_c1 = np.array(d_stop_c1, dtype=object)
d_stop_c2 = np.array(d_stop_c2, dtype=object)
res_cue1 = np.array(res_cue1, dtype=object)
res_cue2 = np.array(res_cue2, dtype=object)
r1_wrong_only = np.array(r1_wrong_only, dtype=object)
r2_wrong_only = np.array(r2_wrong_only, dtype=object)

idx1 = np.argsort(res_cue1[:, 0])
idx2 = np.argsort(res_cue2[:, 0])

x1, y1 = res_cue1[idx1, 0], res_cue1[idx1, 1].astype(float)
x2, y2 = res_cue2[idx2, 0], res_cue2[idx2, 1].astype(float)
x3, y3 = d_stop_c1[idx1, 0], d_stop_c1[idx1, 1].astype(float)
x4, y4 = d_stop_c2[idx2, 0], d_stop_c2[idx2, 1].astype(float)

x5y5 = np.array(r1_stop, dtype=object)
x5, y5 = x5y5[:, 0], x5y5[:, 1].astype(float)

x6y6 = np.array(r2_stop, dtype=object)
x6, y6 = x6y6[:, 0], x6y6[:, 1].astype(float)

x7, y7 = r1_wrong_only[:, 0], r1_wrong_only[:, 1].astype(float)
x8, y8 = r2_wrong_only[:, 0], r2_wrong_only[:, 1].astype(float)

# date-based split (STRING COMPARISON)
x_pre_cue = "2024-11-27_16-11"

m5 = x5 <= x_pre_cue
x5, y5 = x5[m5], y5[m5]

m6 = x6 <= x_pre_cue
x6, y6 = x6[m6], y6[m6]

m1 = x1 >= x_pre_cue
x1, y1 = x1[m1], y1[m1]

m2 = x2 >= x_pre_cue
x2, y2 = x2[m2], y2[m2]

m3 = x3 >= x_pre_cue
x3, y3 = x3[m3], y3[m3]

m4 = x4 >= x_pre_cue
x4, y4 = x4[m4], y4[m4]

# average expert performance (Cue 1 and Cue 2) per session
cue1_expert = pd.Series(res_cue1[:, 1].astype(float), index=res_cue1[:, 0], name="cue1_expert")
cue2_expert = pd.Series(res_cue2[:, 1].astype(float), index=res_cue2[:, 0], name="cue2_expert")
avg_expert = pd.concat([cue1_expert, cue2_expert], axis=1).mean(axis=1).sort_index()
x_avg_expert = avg_expert.index.to_numpy()
y_avg_expert = avg_expert.to_numpy(dtype=float)
m_avg = x_avg_expert >= x_pre_cue
x_avg_expert, y_avg_expert = x_avg_expert[m_avg], y_avg_expert[m_avg]

# plotting
fig = go.Figure()

fig.add_trace(go.Scatter(x=x5, y=y5, mode="lines+markers",
    name="R1 Stop Overall",
    line=dict(width=2, color="lightgrey"),
    marker=dict(size=7, color="lightgrey"),
))

fig.add_trace(go.Scatter(x=x6, y=y6, mode="lines+markers",
    name="R2 Stop Overall",
    line=dict(width=2, color="dimgrey"),
    marker=dict(size=7, color="dimgrey"),
))

fig.add_trace(go.Scatter(x=x1, y=y1, mode="lines+markers",
    name="Cue 1 - Expert",
    line=dict(width=2, color="orange"),
    marker=dict(size=7, color="orange"),
))

fig.add_trace(go.Scatter(x=x3, y=y3, mode="lines+markers",
    name="Cue 1 - Double Stop",
    line=dict(width=2, color="peachpuff", dash="dash"),
    marker=dict(size=7, color="peachpuff"),
))

fig.add_trace(go.Scatter(x=x2, y=y2, mode="lines+markers",
    name="Cue 2 - Expert",
    line=dict(width=2, color="purple"),
    marker=dict(size=7, color="purple"),
))

fig.add_trace(go.Scatter(x=x4, y=y4, mode="lines+markers",
    name="Cue 2 - Double Stop",
    line=dict(width=2, color="lavender", dash="dash"),
    marker=dict(size=7, color="lavender"),
))

# fig.add_trace(go.Scatter(x=x_avg_expert, y=y_avg_expert, mode="lines+markers",
#     name="Average Expert",
#     line=dict(width=3, color="red", dash="dot"),
#     marker=dict(size=7, color="red"),
# ))

# Assembly012 overlay intentionally disabled in this copy.
# fig.add_trace(go.Scatter(x=x_ens012, y=y_ens012, mode="lines+markers",
#     name="Assembly012 Mean Activation",
#     line=dict(width=2, color="blue"),
#     marker=dict(size=6, color="blue"),
#     yaxis="y2",
# ))

all_x_ticks = np.sort(np.unique(np.concatenate([x5, x6, x1, x2, x3, x4, x_avg_expert])))
all_x_ticklabels = [s.split("_")[0] for s in all_x_ticks]

# layout
fig.update_layout(
    title=dict(text="Animal Performance (Expert Trials and Double Stops)", font=dict(size=10, family="Arial")),
    width=900,
    height=340,
    margin=dict(l=55, r=30, t=35, b=95),
    xaxis_title="Session",
    yaxis_title="Proportion of trials (%)",
    plot_bgcolor="white",
    font=dict(family="Arial", size=9),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(0,0,0,0)",
        xanchor="left", yanchor="top",
        orientation="h",
        entrywidth=170,
        entrywidthmode="pixels",
        font=dict(size=9, family="Arial"),
        tracegroupgap=0,
    ),
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.2)",
    zeroline=False,
    showline=True,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)

fig.update_xaxes(
    tickmode="array",
    tickvals=all_x_ticks,
    ticktext=all_x_ticklabels,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.1)",
    showline=True,
    zeroline=False,
    tickangle=-45,
    tickfont=dict(size=9, family="Arial"),
    title_font=dict(size=10, family="Arial"),
)
fig.update_traces(marker=dict(size=6))

fig.show()
